#Bài 1

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

# Kết nối database trong bộ nhớ (hoặc file .db)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 1. Tạo bảng và chèn dữ liệu
cursor.executescript('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);

CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);

INSERT INTO course VALUES (12, 'Giai tich'), (34, 'Thong ke'), (26, 'Tin hoc');

INSERT INTO student (student_id, name, class, course_id, score) VALUES
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
(3, 'Pham Van Nam', 'Toan Tin', NULL, 7.9),
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
(7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
(8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
(9, 'Duong Huu Phuc', 'Kinh Te', NULL, 7.2),
(10, 'Cao Thi Hanh', 'May Tinh', NULL, 7.0);
''')

def run_query(query):
    return pd.read_sql_query(query, conn)

print("Đã khởi tạo dữ liệu thành công!")

Đã khởi tạo dữ liệu thành công!


In [2]:
# a. Tích Descartes (Cross Join)
print("--- Tích Descartes ---")
display(run_query("SELECT * FROM student CROSS JOIN course"))

# b. Các loại Join
print("--- INNER JOIN ---")
display(run_query("SELECT * FROM student INNER JOIN course ON student.course_id = course.id"))

print("--- LEFT JOIN ---")
display(run_query("SELECT * FROM student LEFT JOIN course ON student.course_id = course.id"))

try:
    print("--- RIGHT JOIN ---")
    display(run_query("SELECT * FROM student RIGHT JOIN course ON student.course_id = course.id"))
except:
    print("Môi trường SQLite này chưa hỗ trợ RIGHT JOIN trực tiếp.")

--- Tích Descartes ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


--- INNER JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


--- LEFT JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


--- RIGHT JOIN ---
Môi trường SQLite này chưa hỗ trợ RIGHT JOIN trực tiếp.


#Bài 2

In [3]:
# Xử lý dữ liệu: Chỉ giữ lại các student có course_id hợp lệ trong bảng course
# HOẶC cập nhật lại (theo ý đề bài là làm sạch bảng)
cursor.execute("DELETE FROM student WHERE course_id NOT IN (SELECT id FROM course) AND course_id IS NOT NULL")
conn.commit()

# a. Thống kê theo lớp
query_2a = """
SELECT class, COUNT(student_id) as total_students, AVG(score) as avg_score
FROM student
GROUP BY class
"""
print("Thống kê theo lớp:")
display(run_query(query_2a))

# b. Thống kê theo môn học
query_2b = """
SELECT c.course_name, COUNT(s.student_id) as total_students, AVG(s.score) as avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
"""
print("Thống kê theo môn học:")
display(run_query(query_2b))

# c. Phân loại thi đua
query_2c = """
SELECT name, score,
       CASE
            WHEN score >= 9.0 THEN 'Xuat sac'
            WHEN score >= 6.0 THEN 'Tot'
            ELSE 'Kem'
       END as classification
FROM student
"""
display(run_query(query_2c))

Thống kê theo lớp:


,class,total_students,avg_score
0,Kinh Te,3,8.533333
1,May Tinh,2,6.850000
2,Toan Tin,1,7.900000


Thống kê theo môn học:


,course_name,total_students,avg_score
0,Giai tich,1,6.7
1,Thong ke,2,9.2


,name,score,classification
0,Nguyen Minh Hoang,6.7,Tot
1,Tran Thi Lan,9.2,Xuat sac
2,Pham Van Nam,7.9,Tot
3,Bui Tien Dung,9.2,Xuat sac
4,Duong Huu Phuc,7.2,Tot
5,Cao Thi Hanh,7.0,Tot


#Bài 3

In [4]:
# Xếp hạng tổng quát, theo lớp và theo môn
query_3 = """
SELECT name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) as rank_all,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) as rank_class,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) as rank_course
FROM student
"""
df_rank = run_query(query_3)
display(df_rank)

# Top 3 cao nhất và thấp nhất (Ví dụ cho toàn trường)
print("Top 3 điểm cao nhất:")
display(df_rank.nsmallest(3, 'rank_all'))

print("Top 3 điểm thấp nhất:")
display(df_rank.nlargest(3, 'rank_all'))

,name,class,course_id,score,rank_all,rank_class,rank_course
0,Tran Thi Lan,Kinh Te,34.0,9.2,1,1,1
1,Bui Tien Dung,Kinh Te,34.0,9.2,1,1,1
2,Pham Van Nam,Toan Tin,NaN,7.9,3,1,1
3,Duong Huu Phuc,Kinh Te,NaN,7.2,4,3,2
4,Cao Thi Hanh,May Tinh,NaN,7.0,5,1,3
5,Nguyen Minh Hoang,May Tinh,12.0,6.7,6,2,1


Top 3 điểm cao nhất:


,name,class,course_id,score,rank_all,rank_class,rank_course
0,Tran Thi Lan,Kinh Te,34.0,9.2,1,1,1
1,Bui Tien Dung,Kinh Te,34.0,9.2,1,1,1
2,Pham Van Nam,Toan Tin,NaN,7.9,3,1,1


Top 3 điểm thấp nhất:


,name,class,course_id,score,rank_all,rank_class,rank_course
5,Nguyen Minh Hoang,May Tinh,12.0,6.7,6,2,1
4,Cao Thi Hanh,May Tinh,NaN,7.0,5,1,3
3,Duong Huu Phuc,Kinh Te,NaN,7.2,4,3,2


#Bài 4

In [5]:
# Thêm cột mới
try:
    cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
except:
    pass # Nếu đã tồn tại cột

# Cập nhật ngày tốt nghiệp dựa trên điểm số
# Công thức: Today + (score * 30 ngày)
current_date = datetime.now()

# Lấy dữ liệu ra xử lý bằng Python cho linh hoạt với DATETIME
students = cursor.execute("SELECT student_id, score FROM student").fetchall()
for sid, score in students:
    if score:
        grad_date = current_date + timedelta(days=int(score * 30))
        cursor.execute("UPDATE student SET graduation_date = ? WHERE student_id = ?", (grad_date.strftime('%Y-%m-%d %H:%M:%S'), sid))

conn.commit()

print("Bảng student sau khi cập nhật ngày tốt nghiệp:")
display(run_query("SELECT student_id, name, score, graduation_date FROM student"))

Bảng student sau khi cập nhật ngày tốt nghiệp:


,student_id,name,score,graduation_date
0,1,Nguyen Minh Hoang,6.7,2026-10-21 07:41:13
1,2,Tran Thi Lan,9.2,2027-01-04 07:41:13
2,3,Pham Van Nam,7.9,2026-11-26 07:41:13
3,7,Bui Tien Dung,9.2,2027-01-04 07:41:13
4,9,Duong Huu Phuc,7.2,2026-11-05 07:41:13
5,10,Cao Thi Hanh,7.0,2026-10-30 07:41:13
